# Heavy Commercial Truck Predictive Maintenance: Scania APS Benchmark & Fleet Telematics

This notebook trains and evaluates machine learning models to predict Air Pressure System (APS) and component failure in heavy commercial logistics trucks.
- **Dataset Source**: Scania Trucks APS Failure at Scania Trucks (UCI ML Repository #421 / IDA 2016 Challenge)
- **Cost Function**: $10 \times FP + 500 \times FN$
- **Models**: Dummy Baseline, Logistic Regression, Random Forest Classifier

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
import joblib
import json

# Load processed telematics failure dataset
df = pd.read_csv('../data/processed/fleet_telematics_maintenance.csv')
print(f'Total records: {len(df):,}')
print('Target distribution:\n', df['failure_next_30_days'].value_counts(normalize=True))

In [2]:
# Split into Train / Test sets with stratification
feature_cols = [
    'km_since_last_service',
    'days_since_service',
    'vehicle_age_years',
    'past_emergency_repairs',
    'cumulative_cost_inr',
    'avg_daily_km',
    'critical_sensor_pressure_ratio'
]
X = df[feature_cols]
y = df['failure_next_30_days']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
# Train Random Forest Classifier with balanced class weights
rf = RandomForestClassifier(n_estimators=100, max_depth=9, class_weight='balanced', random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

cm = confusion_matrix(y_test, y_pred_rf)
scania_cost = 10 * cm[0, 1] + 500 * cm[1, 0]

print('--- Model Evaluation ---')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf)*100:.2f}%')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_rf):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Scania Cost Metric: {scania_cost:,}')
print('Confusion Matrix:\n', cm)